# Clean & Prepare Event Log

## import and setup

In [19]:
import pandas as pd
import numpy as np
import pm4py

from pathlib import Path
import yaml

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [20]:
ROOT = Path().resolve().parents[0]
SCHEMA = ROOT / "configs/schema.yaml"
STAGING = ROOT / "data/staging/snapshots.parquet"

In [21]:
df = pd.read_parquet(STAGING)

schema = yaml.safe_load(open(SCHEMA, "r", encoding="utf-8"))
col_map = schema['raw_to_canonical']

## data understanding

In [22]:
df.head(7)

,case_id,case_status,is_open,reassignment_total,reopen_total,system_update_count,met_deadline,affected_user_id,reported_ref_id,opened_at,created_by_str_id,created_at,updated_by_str_id,updated_at,contact_channel,location_id,category,subcategory,reported_symptom,asset_id,impact_level,urgency_level,computed_priority_level,assigned_team_id,assigned_agent_id,used_knowledge_base,priority_confirmed,notify_email,issue_id,change_request_id,vendor_id,caused_by_ref_id,closed_code,resolved_by_agent_id,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,2016-02-29 01:23:00,Updated by 21,2016-02-29 01:23:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,None,2 - Medium,2 - Medium,3 - Moderate,Group 56,None,True,False,False,None,None,None,None,code 5,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,2016-02-29 01:23:00,Updated by 642,2016-02-29 08:53:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,None,2 - Medium,2 - Medium,3 - Moderate,Group 56,None,True,False,False,None,None,None,None,code 5,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,2016-02-29 01:23:00,Updated by 804,2016-02-29 11:29:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,None,2 - Medium,2 - Medium,3 - Moderate,Group 56,None,True,False,False,None,None,None,None,code 5,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,2016-02-29 01:23:00,Updated by 908,2016-03-05 12:00:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,None,2 - Medium,2 - Medium,3 - Moderate,Group 56,None,True,False,False,None,None,None,None,code 5,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,Created by 171,2016-02-29 04:57:00,Updated by 746,2016-02-29 04:57:00,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,None,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,False,None,None,None,None,code 5,Resolved by 81,2016-03-01 09:52:00,2016-03-06 10:00:00
5,INC0000047,Active,True,1,0,1,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,Created by 171,2016-02-29 04:57:00,Updated by 21,2016-02-29 05:30:00,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,None,2 - Medium,2 - Medium,3 - Moderate,Group 24,Resolver 31,True,False,False,None,None,None,None,code 5,Resolved by 81,2016-03-01 09:52:00,2016-03-06 10:00:00
6,INC0000047,Active,True,1,0,2,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,Created by 171,2016-02-29 04:57:00,Updated by 21,2016-02-29 05:33:00,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,None,2 - Medium,2 - Medium,3 - Moderate,Group 24,Resolver 31,True,False,False,None,None,None,None,code 5,Resolved by 81,2016-03-01 09:52:00,2016-03-06 10:00:00


In [23]:
df = df.sort_values(by=['case_id', 'updated_at', 'system_update_count'])

In [24]:
df.dtypes

case_id                            object
case_status                        object
is_open                              bool
reassignment_total                  int64
reopen_total                        int64
system_update_count                 int64
met_deadline                         bool
affected_user_id                   object
reported_ref_id                    object
opened_at                  datetime64[ns]
created_by_str_id                  object
created_at                 datetime64[ns]
updated_by_str_id                  object
updated_at                 datetime64[ns]
contact_channel                    object
location_id                        object
category                           object
subcategory                        object
reported_symptom                   object
asset_id                           object
impact_level                       object
urgency_level                      object
computed_priority_level            object
assigned_team_id                  

In [25]:
df.select_dtypes(include='object').nunique().sort_values()

impact_level                   3
urgency_level                  3
caused_by_ref_id               3
computed_priority_level        4
vendor_id                      4
contact_channel                5
case_status                    9
closed_code                   17
asset_id                      50
category                      58
assigned_team_id              78
change_request_id            181
created_by_str_id            185
reported_ref_id              207
resolved_by_agent_id         216
location_id                  224
assigned_agent_id            234
issue_id                     252
subcategory                  254
reported_symptom             525
updated_by_str_id            846
affected_user_id            5244
case_id                    24918
dtype: int64

In [26]:
df.select_dtypes(include='datetime').nunique().sort_values()

closed_at       2707
created_at     11552
resolved_at    18505
opened_at      19849
updated_at     50664
dtype: int64

In [27]:
df.select_dtypes(include='int').nunique().sort_values()

reopen_total             9
reassignment_total      28
system_update_count    115
dtype: int64

## fix met_deadline as final outcome

In [28]:
# event level count of met_deadline
df['met_deadline'].value_counts()

met_deadline
True     132497
False      9215
Name: count, dtype: int64

In [29]:
# no of cases in data
df['case_id'].nunique()

24918

In [30]:
# no of cases starting with true met_deadline
df.groupby('case_id')['met_deadline'].first().sum()

np.int64(24917)

In [31]:
# no of cases that last entry with false in met_deadline
(~df.groupby('case_id')['met_deadline'].last()).sum()

np.int64(9115)

In [32]:
# cases with change in met_deadline 
(df.groupby('case_id')['met_deadline'].nunique() > 1).sum()

np.int64(9114)

This implies that deadline status is fliping over timesteps, and that met_deadline is tracking the deadline status at that time. 

Only 1 case starting with false met_deadline (which didn't met the deadline from start) [24917 only true first] and 1 extra case [9115 vs 9114] has false.

In [33]:
# case that initlized with false met_deadline
met = ~df.groupby('case_id')['met_deadline'].first()
met[met].index

Index(['INC0002588'], dtype='object', name='case_id')

In [34]:
df[df['case_id'] == 'INC0002588']

,case_id,case_status,is_open,reassignment_total,reopen_total,system_update_count,met_deadline,affected_user_id,reported_ref_id,opened_at,created_by_str_id,created_at,updated_by_str_id,updated_at,contact_channel,location_id,category,subcategory,reported_symptom,asset_id,impact_level,urgency_level,computed_priority_level,assigned_team_id,assigned_agent_id,used_knowledge_base,priority_confirmed,notify_email,issue_id,change_request_id,vendor_id,caused_by_ref_id,closed_code,resolved_by_agent_id,resolved_at,closed_at
11347,INC0002588,Active,True,0,0,0,False,Caller 3571,Opened by 325,2016-03-04 15:00:00,Created by 140,2016-03-04 15:00:00,Updated by 614,2016-03-04 15:00:00,Phone,Location 83,Category 55,Subcategory 270,Symptom 208,None,2 - Medium,2 - Medium,3 - Moderate,Group 33,Resolver 159,False,False,False,None,None,None,None,code 1,Resolved by 143,2016-03-11 10:38:00,2016-03-24 18:50:00
11348,INC0002588,Awaiting Problem,True,0,0,1,False,Caller 3571,Opened by 325,2016-03-04 15:00:00,Created by 140,2016-03-04 15:00:00,Updated by 614,2016-03-04 17:32:00,Phone,Location 83,Category 55,Subcategory 270,Symptom 208,None,2 - Medium,2 - Medium,3 - Moderate,Group 33,Resolver 159,False,False,False,None,None,None,None,code 1,Resolved by 143,2016-03-11 10:38:00,2016-03-24 18:50:00
11349,INC0002588,Resolved,True,0,0,2,False,Caller 3571,Opened by 325,2016-03-04 15:00:00,Created by 140,2016-03-04 15:00:00,Updated by 614,2016-03-11 10:38:00,Phone,Location 83,Category 55,Subcategory 270,Symptom 208,None,2 - Medium,2 - Medium,3 - Moderate,Group 33,Resolver 159,False,False,False,None,None,None,None,code 1,Resolved by 143,2016-03-11 10:38:00,2016-03-24 18:50:00
11350,INC0002588,Closed,False,0,0,3,False,Caller 3571,Opened by 325,2016-03-04 15:00:00,Created by 140,2016-03-04 15:00:00,Updated by 908,2016-03-24 18:50:00,Phone,Location 83,Category 55,Subcategory 270,Symptom 208,None,2 - Medium,2 - Medium,3 - Moderate,Group 33,Resolver 159,False,False,False,None,None,None,None,code 1,Resolved by 143,2016-03-11 10:38:00,2016-03-24 18:50:00


In [35]:
df['met_deadline'] = df.groupby('case_id')['met_deadline'].transform('last')

## time based features

In [36]:
time_df = df[schema['timestamp_cols']]

In [37]:
time_col_props = time_df.describe()
time_col_props.loc['range'] = (time_col_props.loc['max'] - time_col_props.loc['min'])
time_col_props

,opened_at,created_at,updated_at,resolved_at,closed_at
count,141712,88636,141712,138571,141712
mean,2016-04-12 22:19:09.100852736,2016-04-08 16:21:18.906539008,2016-04-19 10:49:49.286863616,2016-04-24 04:55:32.415873280,2016-04-29 23:48:41.171248640
min,2016-02-29 01:16:00,2016-02-29 01:23:00,2016-02-29 01:23:00,2016-02-29 09:04:00,2016-02-29 17:47:00
25%,2016-03-16 15:24:00,2016-03-14 16:13:00,2016-03-24 08:36:45,2016-03-25 09:17:30,2016-03-31 09:59:00
50%,2016-04-07 16:27:00,2016-04-01 20:08:00,2016-04-13 11:06:00,2016-04-15 13:09:00,2016-04-20 13:07:00
75%,2016-05-04 10:32:00,2016-04-28 16:43:00,2016-05-09 14:02:00,2016-05-12 12:36:00,2016-05-17 13:07:00
max,2017-02-16 14:17:00,2017-01-27 16:59:00,2017-02-18 15:00:00,2017-02-17 00:47:00,2017-02-18 15:00:00
range,353 days 13:01:00,333 days 15:36:00,355 days 13:37:00,353 days 15:43:00,354 days 21:13:00


In [38]:
time_df.nunique().sort_values()

closed_at       2707
created_at     11552
resolved_at    18505
opened_at      19849
updated_at     50664
dtype: int64

In [39]:
time_df.isna().sum().sort_values()

opened_at          0
updated_at         0
closed_at          0
resolved_at     3141
created_at     53076
dtype: int64

In [40]:
df.groupby('case_id')[schema['timestamp_cols']].apply(lambda case_values: case_values.isna().all()).sum()

opened_at          0
created_at     11495
updated_at         0
resolved_at     1556
closed_at          0
dtype: int64

In [41]:
df.groupby('case_id')[schema['timestamp_cols']].apply(lambda case_values: case_values.isna().any()).sum()

opened_at          0
created_at     11495
updated_at         0
resolved_at     1556
closed_at          0
dtype: int64

If the created and resolved time is unknown, its unknown for all its events. -> meaning it can not be inferred and/or imputed from its past/future events.

created_at timestamp is missing for 40-50% cases, while resolved_at is missing for around 7% cases.

Now, checking anomaly/unusual situations. Only opened, closed and updated timestamp have no unknown values.

In [42]:
group_time_obj = df.groupby('case_id')[schema['timestamp_cols']]

In [43]:
group_time_obj.first()

,opened_at,created_at,updated_at,resolved_at,closed_at
case_id,,,,,
INC0000045,2016-02-29 01:16:00,2016-02-29 01:23:00,2016-02-29 01:23:00,2016-02-29 11:29:00,2016-03-05 12:00:00
INC0000047,2016-02-29 04:40:00,2016-02-29 04:57:00,2016-02-29 04:57:00,2016-03-01 09:52:00,2016-03-06 10:00:00
INC0000057,2016-02-29 06:10:00,NaT,2016-02-29 06:26:00,2016-03-01 02:55:00,2016-03-06 03:00:00
INC0000060,2016-02-29 06:38:00,2016-02-29 06:42:00,2016-02-29 06:42:00,2016-03-02 12:06:00,2016-03-07 13:00:00
INC0000062,2016-02-29 06:58:00,2016-02-29 07:26:00,2016-02-29 07:26:00,2016-02-29 15:51:00,2016-03-05 16:00:00
...,...,...,...,...,...
INC0120304,2017-02-15 02:02:00,NaT,2017-02-15 02:02:00,2017-02-17 00:47:00,2017-02-17 00:50:00
INC0120319,2017-02-15 07:09:00,NaT,2017-02-15 07:09:00,NaT,2017-02-15 07:09:00
INC0120495,2017-02-15 11:58:00,NaT,2017-02-15 11:58:00,NaT,2017-02-16 09:51:00


In [44]:
print("\n#entries - closed before opening:", (time_df['closed_at'] < time_df['opened_at']).sum(), 
      "\n#entries - closed before updating:", (time_df['closed_at'] < time_df['updated_at']).sum(),
      "\n#entries - updated before opening:", (time_df['updated_at'] < time_df['opened_at']).sum(),
)


#entries - closed before opening: 0 
#entries - closed before updating: 0 
#entries - updated before opening: 5


### fixing opened_at

In [45]:
anomaly_open_at = df[df['updated_at'] < df['opened_at']]['case_id'].values
anomaly_open_at

array(['INC0000130', 'INC0000131', 'INC0000134', 'INC0000135',
       'INC0000164'], dtype=object)

In [46]:
case_and_time = schema['timestamp_cols'].copy()
case_and_time.insert(0, 'case_id')
case_and_time

['case_id',
 'opened_at',
 'created_at',
 'updated_at',
 'resolved_at',
 'closed_at']

In [47]:
df[df['case_id'].isin(anomaly_open_at)][case_and_time].head()

,case_id,opened_at,created_at,updated_at,resolved_at,closed_at
328,INC0000130,2016-02-29 09:41:00,2016-02-29 07:17:00,2016-02-29 07:27:00,2016-03-03 10:16:00,2016-03-08 11:00:00
329,INC0000130,2016-02-29 09:41:00,2016-02-29 07:17:00,2016-02-29 09:59:00,2016-03-03 10:16:00,2016-03-08 11:00:00
330,INC0000130,2016-02-29 09:41:00,2016-02-29 07:17:00,2016-03-01 14:13:00,2016-03-03 10:16:00,2016-03-08 11:00:00
331,INC0000130,2016-02-29 09:41:00,2016-02-29 07:17:00,2016-03-03 10:16:00,2016-03-03 10:16:00,2016-03-08 11:00:00
332,INC0000130,2016-02-29 09:41:00,2016-02-29 07:17:00,2016-03-03 10:16:00,2016-03-03 10:16:00,2016-03-08 11:00:00


such instances might occur due to system error, different timezones, script in servicenow.

The reported user median will be taken time wil be imputed the anomaly. reported_ref_id is considered because that id is the one who opened the case.

In [48]:
df[['affected_user_id', 'reported_ref_id', 'created_by_str_id', 'updated_by_str_id']].isna().sum()

affected_user_id        29
reported_ref_id       4835
created_by_str_id    53076
updated_by_str_id        0
dtype: int64

In [49]:
# impute such instances with 
anomaly_open_report_id = df[df['case_id'].isin(anomaly_open_at)]['reported_ref_id'].unique()
anomaly_open_report_id

array(['Opened by  62', 'Opened by  433', 'Opened by  501',
       'Opened by  40', 'Opened by  131'], dtype=object)

In [50]:
# exclude anomaly cases for describe compute
exclude_anomaly_df = df[~df['case_id'].isin(anomaly_open_at)]

exclude_anomaly_df['open_to_create'] = exclude_anomaly_df['created_at'] - exclude_anomaly_df['opened_at']
open_to_create_p75 = exclude_anomaly_df[exclude_anomaly_df['reported_ref_id'].isin(anomaly_open_report_id)].\
    groupby('reported_ref_id')['open_to_create'].quantile(0.75).astype('timedelta64[ns]')

In [51]:
open_to_create_p75.loc['Opened by  131']

Timedelta('0 days 00:13:00')

In [52]:
mask = (df['opened_at'] > df['updated_at'])
df.loc[mask, 'created_at']

328   2016-02-29 07:17:00
334   2016-02-29 07:14:00
344   2016-02-29 07:46:00
349   2016-02-29 07:46:00
436   2016-02-29 08:49:00
Name: created_at, dtype: datetime64[ns]

The system created timestamp is the first update system makes for the case. 

In [53]:
# p75 of diff for all reported_ref_id 
p75_diff = (
    df[df['updated_at'] > df['opened_at']] # use only right values
    .groupby('reported_ref_id')
    .apply(lambda x: (x['created_at'] - x['opened_at']).quantile(0.75))
)

In [54]:
# mapping to our actual df
df['p75'] = df['reported_ref_id'].map(p75_diff)

mask = df['updated_at'] < df['opened_at']
df.loc[mask, 'opened_at'] = df.loc[mask, 'created_at'] - df.loc[mask, 'p75']

df.drop(columns='p75', inplace=True)

good that opened_at has no unknown values now, but in service now incident management system; sla timer starts from the time case is opened which is mostly unknown and halts when case is resolved or on-hold.

### fixing created_at - used workaround
around 40-50% are unknown, instead using opened_at with an assumption that the case open after short while.  

In [63]:
# case_status distribution where created_at is unkown
mask = df.groupby('case_id').first()['created_at'].isna()
df.groupby('case_id').first().loc[mask, 'case_status'].value_counts()

case_status
New                   6450
Active                3438
Resolved              1543
Awaiting User Info      63
Awaiting Problem         1
Name: count, dtype: int64

In [ ]:
# cases where first status is not New but later New as case_status is found.
(df.groupby('case_id').
    apply(lambda case_values: (case_values.iloc[1:]['case_status'] == 'New').any() * (case_values.iloc[0]['case_status'] != 'New'))
).sum()

np.int64(0)

In [50]:
# check assumption in data before impute
mask = df['updated_at'] == df['created_at']
(df[mask]['case_status'] == 'New').value_counts()

case_status
True     11190
False     4570
Name: count, dtype: int64

In [57]:
df[mask][(df[mask]['case_status'] == 'New')]['case_id'].nunique()

9943

there are cases with updated_at and created_at (meaning case is just created and new) is same but case_status is not New 

In [103]:
df['this_update_time'] = df.groupby('case_id')[['updated_at']].diff(1)
df['this_update_time'] = df['this_update_time'].fillna(df['updated_at'] - df['opened_at'])
df['this_update_time']

0        0 days 00:07:00
1        0 days 07:30:00
2        0 days 02:36:00
3        5 days 00:31:00
4        0 days 00:17:00
               ...      
141707   0 days 00:00:00
141708   0 days 00:00:00
141709   0 days 01:03:00
141710   0 days 01:18:00
141711   0 days 00:00:00
Name: this_update_time, Length: 141712, dtype: timedelta64[ns]

In [106]:
df['this_update_time'].isna().sum()

np.int64(0)

In [ ]:
# Check if 'new' rows accumulate significant time
df[df['case_status'] == 'New']['this_update_time'].describe()

count                        36407
mean     0 days 13:35:53.449061993
std      4 days 13:35:50.402121920
min                0 days 00:00:00
25%                0 days 00:00:00
50%                0 days 00:06:00
75%                0 days 01:02:00
max              214 days 05:38:00
Name: this_update_time, dtype: object

In [ ]:
# flag suspicious cases
suspicious = df[(df['case_status'] == 'New') & 
                (df['this_update_time'] > pd.Timedelta('7 days'))]
print(f"Suspicious cases: {suspicious['case_id'].nunique()}")

#the 'New' time correlates with met deadline
df[df['case_status'] == 'New'].groupby(
    pd.cut(df['this_update_time'].dt.days, bins=[0,1,7,30,999])
)['met_deadline'].mean()

Suspicious cases: 441


this_update_time
(0, 1]       0.326819
(1, 7]       0.178415
(7, 30]      0.010563
(30, 999]    0.000000
Name: met_deadline, dtype: float64

In [ ]:
# cateogrize feature — captures the non-linear relationship
df['new_status_urgency'] = pd.cut(
    df[df['case_status'] == 'New']['this_update_time'].dt.days,
    bins=[0, 1, 7, 30, 999],
    labels=['low', 'medium', 'high', 'critical']
)
# 3. Total time spent in 'New' per case as a feature
df[df['case_status'] == 'New'].groupby('case_id')['this_update_time'].sum()

case_id
INC0000045   0 days 00:07:00
INC0000047   0 days 00:17:00
INC0000057   0 days 20:44:00
INC0000060   0 days 00:04:00
INC0000062   0 days 08:53:00
                   ...      
INC0119987   0 days 00:00:00
INC0120268   0 days 00:00:00
INC0120303   0 days 00:00:00
INC0120319   0 days 00:00:00
INC0120495   0 days 00:00:00
Name: this_update_time, Length: 16397, dtype: timedelta64[ns]

In [125]:
# is created_at missing randomly
df['created_at_missing'] = df['created_at'].isna()
df.groupby('created_at_missing')['met_deadline'].mean()

created_at_missing
False    0.421759
True     0.600554
Name: met_deadline, dtype: float64

The deadline show some shift and correlation with missing value of created_at.

### fixing resolved_at

In [ ]:
# majority cases have some delay in closing case.
(df['resolved_at'] != df['closed_at']).sum()

np.int64(141367)

In [ ]:
# check cases where case is not closed
(df.groupby('case_id').last()['case_status'] != 'Closed').sum()

np.int64(0)

In [ ]:
# is resolved same for all entries per case_id
(df.groupby('case_id')['resolved_at'].nunique(dropna=False) <= 1).all()

np.True_

In [170]:
resolved_entries = df['case_status'].eq('Resolved')
mask = df['resolved_at'].isna()

resolved_last_updated = (
    df['updated_at']
      .where(resolved_entries)
      .groupby(df['case_id'])
      .transform('max')
)
df.loc[mask, 'resolved_at'] = resolved_last_updated[mask]

In [171]:
# 0 confirm everything worked well
(df['resolved_at'] > df['closed_at']).sum()

np.int64(0)

In [176]:
mask = df['resolved_at'].isna() # remaining nan
df.loc[mask, 'resolved_at'] = df.loc[mask, 'closed_at'] # cases without Resolved status

In [178]:
df['resolved_at'].isna().sum()

np.int64(0)

### engineering remaining time to resolve the case

In [179]:
df['remaining_time'] = df['resolved_at'] - df['updated_at']

## object/categorical features